[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/bike-availability-data-science-full/blob/main/notebooks/Module_01_Introduction/M1_03_open_data_sources.ipynb)

# 🌐 Module 01: Resource -- Fetch all CityBikes Network and Filter by country

**Purpose**: Fectching available bike-sharing data for Amsterdam

**Module**: Module 01 - Introduction  
**Author**: Ruby van Rooyen  
**Date**: 2025-12-30

### 📋 Overview

In this notebook, you will:
- Fetch a comprehensive list of all available bike-sharing networks worldwide from the CityBikes API.
- Filter these networks to identify and display those specifically located in the Netherlands.
- Access and retrieve real-time station data for a selected bike-sharing network (OV-fiets).
- Filter the retrieved station data to show only stations located in Amsterdam.
- Convert the structured JSON data into a pandas DataFrame for further analysis.

**Goal**: To demonstrate how to programmatically access and process bike-sharing data from open API sources, specifically focusing on filtering and structuring the data for a targeted analysis.

---

## 🔧 Setup

Run this cell first to set up the environment.

In [ ]:
# ═══════════════════════════════════════════════════════════
# NOTEBOOK SETUP
# ═══════════════════════════════════════════════════════════

import sys
import os
from datetime import datetime

# Third-party imports
import pandas as pd
import numpy as np
import requests
import json

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("📍 Running in Google Colab")
    # Uncomment and modify if you need to clone the repository
    # !git clone https://github.com/vinculum3141-ship-it/bike-availability-data-science-full.git
    # %cd bike-availability-data-science-full
else:
    print("📍 Running locally")

print("✅ Setup complete!")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d')}")

📍 Running in Google Colab
✅ Setup complete!
🐍 Python version: 3.12.12
📅 Date: 2026-01-13


## 🚴 Part 2: Bike-Sharing Data (Primary Source #1)

### Why Bike Data?

Bike-sharing systems generate rich, real-time data:
- 📍 **Location** - Station coordinates, neighborhoods
- 🕐 **Time** - Timestamp of every status update
- 🚲 **Availability** - Number of bikes and docks
- 📊 **Patterns** - Usage trends by time/location

This is the **target** for our prediction model!

### Option A: Amsterdam Bike Data

**Source**: City of Amsterdam Open Data Portal

**Details:**
- **URL**: https://data.amsterdam.nl/
- **API Endpoint**: Available through Amsterdam Data Portal
- **Update Frequency**: Real-time (every few minutes)
- **Coverage**: All bike-sharing stations in Amsterdam
- **License**: Open data (CC0 or similar)

**Data Fields:**
```python
{
    "station_id": "AMS-001",
    "station_name": "Centraal Station",
    "latitude": 52.3791,
    "longitude": 4.9003,
    "bikes_available": 12,
    "docks_available": 8,
    "total_capacity": 20,
    "timestamp": "2024-01-15T14:30:00Z",
    "is_active": true
}
```

#### How to Search the City of Amsterdam Open Data Portal

1.  **Go to the Portal Website**: Open your web browser and navigate to the [City of Amsterdam Open Data Portal](https://data.amsterdam.nl/).
2.  **Use the Search Bar**: Look for a search bar or a "Datasets" section. Try searching for terms like:
    *   "fiets" (Dutch for bicycle)
    *   "fietsstations" (bicycle stations)
    *   "deelfietsen" (shared bicycles)
    *   "bike sharing"
    *   "mobiliteit" (mobility)
3.  **Explore Categories**: If there are data categories, look for "Traffic & Transport," "Mobility," or similar.
4.  **Look for API Documentation**: Once you find a relevant dataset, check if it offers an API. There's usually a link to "API Documentation," "Developer Info," or an "Access Data" section that will provide the API endpoint URL and how to use it.

#### Example API Call (Once you find an endpoint)

If you find an API endpoint for bike stations from the Amsterdam Data Portal, you can use a code structure similar to the one below. **You will need to replace `your_amsterdam_api_endpoint_url` with the actual URL you find.**

In [ ]:
import requests
import pandas as pd

# Placeholder for the actual API endpoint you find on data.amsterdam.nl
# --- REPLACE THIS URL WITH THE ACTUAL API ENDPOINT YOU FIND ---
amsterdam_data_api_url = "https://api.data.amsterdam.nl/v1/bike-stations/example/"
# --------------------------------------------------------------

print(f"Attempting to fetch data from: {amsterdam_data_api_url}")

try:
    response = requests.get(amsterdam_data_api_url, timeout=10)
    response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)

    amsterdam_bike_data = response.json()

    print("✅ Successfully fetched data from Amsterdam Open Data Portal!")
    # Depending on the structure of the API response, you might need to adjust this
    # For example, if the data is nested under a key like 'results' or 'stations'
    if isinstance(amsterdam_bike_data, list):
        df_amsterdam = pd.DataFrame(amsterdam_bike_data)
    elif isinstance(amsterdam_bike_data, dict) and 'stations' in amsterdam_bike_data:
        df_amsterdam = pd.DataFrame(amsterdam_bike_data['stations'])
    else:
        print("💡 Data structure not immediately recognized. Displaying raw JSON keys:")
        print(amsterdam_bike_data.keys())
        df_amsterdam = pd.DataFrame([amsterdam_bike_data]) # Attempt to create DataFrame from entire dict

    if not df_amsterdam.empty:
        print("\n📊 Sample Data (first 5 rows):\n")
        print(df_amsterdam.head())
        print(f"\nTotal records fetched: {len(df_amsterdam)}")
    else:
        print("No data rows were extracted.")

except requests.exceptions.RequestException as e:
    print(f"❌ Error fetching data from Amsterdam Open Data Portal: {e}")
    print("Please ensure the URL is correct and the API is accessible.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")
    print("Consider inspecting the raw `amsterdam_bike_data` to understand its structure.")


### Option B: CityBikes API (Global)

**Source**: CityBikes - Global Bike Sharing Data

**Details:**
- **URL**: http://api.citybik.es/v2/
- **Coverage**: 600+ cities worldwide
- **Authentication**: None required! 🎉
- **Update Frequency**: Real-time (varies by city)
- **License**: Open data

**Available Cities Include:**
- Amsterdam (Netherlands)
- London (UK)
- New York City (USA)
- Paris (France)
- Barcelona (Spain)
- And many more!

**Why CityBikes?**
- ✅ No API key needed
- ✅ Simple JSON format
- ✅ Standardized across cities
- ✅ Great for learning!


#### Insert Code to Fetch All CityBikes Networks

Python code to fetch the list of all available bike-sharing networks worldwide from the CityBikes API '/networks' endpoint.
* Perform an HTTP GET request to the CityBikes API '/networks' endpoint,
* handle potential errors, and
* print the number of networks found

In [ ]:
print("🌐 Fetching all bike-sharing networks from CityBikes API...")

networks_url = "http://api.citybik.es/v2/networks"

try:
    # Make API request to get all networks
    response_all_networks = requests.get(networks_url, timeout=10)
    response_all_networks.raise_for_status()  # Raise error for bad status codes

    # Parse JSON response
    all_networks_data = response_all_networks.json()

    # Extract the list of networks
    networks_list = all_networks_data.get('networks', [])

    print(f"✅ Successfully fetched data! Found {len(networks_list)} bike-sharing networks worldwide.")
    print("You can inspect 'networks_list' variable for details.")

except requests.exceptions.RequestException as e:
    print(f"❌ Error fetching all networks data: {e}")
    networks_list = [] # Initialize an empty list in case of error
    print("   Skipping network data fetching for now.")


🌐 Fetching all bike-sharing networks from CityBikes API...
✅ Successfully fetched data! Found 793 bike-sharing networks worldwide.
You can inspect 'networks_list' variable for details.


#### Insert Code to Fetch All CityBikes Networks in the Netherlands
Now that the `networks_list` is populated, filter this list to identify bike-sharing networks specifically located in the Netherlands
* Display their names, IDs, and cities

In [ ]:
print("🇳🇱 Filtering for networks in the Netherlands...")

nl_networks = []

# Ensure networks_list is available (from previous cell or initialized as empty)
if 'networks_list' not in locals() or not networks_list:
    print("❌ networks_list not available or empty. Cannot filter.")
else:
    for network in networks_list:
        # Check if the network has location data and if the country is 'NL'
        if 'location' in network and network['location'].get('country') == 'NL':
            nl_networks.append(network)

    if nl_networks:
        print(f"✅ Found {len(nl_networks)} bike-sharing network(s) in the Netherlands:")
        for network in nl_networks:
            city_name = network['location'].get('city', 'N/A')
            display_name = f"* {network['name']} (ID: {network['id']}), City: {city_name}"
            print(display_name)
    else:
        print("No bike-sharing networks found in the Netherlands.")

🇳🇱 Filtering for networks in the Netherlands...
✅ Found 2 bike-sharing network(s) in the Netherlands:
* Cykl (ID: cykl), City: Wageningen
* OV-fiets (ID: ov-fiets), City: Nederland


In [ ]:
# ═══════════════════════════════════════════════════════════
# DEMO: FETCH BIKE DATA FROM CITYBIKES API
# ═══════════════════════════════════════════════════════════

print("🚴 Fetching bike-sharing data from CityBikes API...\n")

#Using an OV-fiets (public transport bike), you must return the bike to the specific station location where you rented it.
#If no docks are available, just lock securely and return the key to the drop box
url = "http://api.citybik.es/v2/networks/ov-fiets"


try:
    # Make API request
    response = requests.get(url, timeout=10)
    response.raise_for_status()  # Raise error for bad status codes

    # Parse JSON response
    data = response.json()

    # Extract basic info
    network_info = data['network']
    print(f"✅ Successfully fetched data!")
    print(f"\n📊 Network Information:")
    print(f"   Name: {network_info['name']}")
    print(f"   City: {network_info['location']['city']}")
    print(f"   Country: {network_info['location']['country']}")
    print(f"   Total Stations: {len(network_info['stations'])}")

    # Look at first few stations
    print(f"\n🔍 Sample Stations (first 3):\n")
    for station in network_info['stations'][:3]:
        print(f"   Station: {station['name']}")
        print(f"      Location: ({station['latitude']:.4f}, {station['longitude']:.4f})")
        print(f"      Bikes available: {station['free_bikes']}")
        print(f"      Docks available: {station['empty_slots']}")
        print(f"      Timestamp: {station['timestamp']}")
        print()

    print("=" * 60)
    print("🎉 API demo successful! We can fetch real-time bike data!")
    print("=" * 60)

except requests.exceptions.RequestException as e:
    print(f"❌ Error fetching data: {e}")
    print("   Don't worry! We'll work with sample data instead.")

🚴 Fetching bike-sharing data from CityBikes API...

✅ Successfully fetched data!

📊 Network Information:
   Name: OV-fiets
   City: Nederland
   Country: NL
   Total Stations: 284

🔍 Sample Stations (first 3):

   Station: Alkmaar
      Location: (52.6375, 4.7398)
      Bikes available: 57
      Docks available: None
      Timestamp: 2025-12-15T14:51:22.740838+00:00Z

   Station: Amsterdam Noord
      Location: (52.4024, 4.9310)
      Bikes available: 7
      Docks available: None
      Timestamp: 2025-12-15T14:51:22.740364+00:00Z

   Station: Utrecht Terwijde
      Location: (52.1000, 5.0441)
      Bikes available: 4
      Docks available: None
      Timestamp: 2025-12-15T14:51:22.740143+00:00Z

🎉 API demo successful! We can fetch real-time bike data!


In [ ]:
print("🔍 Filtering OV-fiets stations for Amsterdam...")

amsterdam_ov_stations = []

if 'network_info' in locals() and network_info.get('stations'):
    for station in network_info['stations']:
        if 'name' in station and 'Amsterdam' in station['name']:
            amsterdam_ov_stations.append(station)

    if amsterdam_ov_stations:
        print(f"✅ Found {len(amsterdam_ov_stations)} OV-fiets station(s) with 'Amsterdam' in their name:")
        for station in amsterdam_ov_stations:
            print(f"   - Station: {station['name']} (ID: {station['id']})")
            print(f"     Location: ({station['latitude']:.4f}, {station['longitude']:.4f})")
            print(f"     Bikes available: {station['free_bikes']}")
            print(f"     Docks available: {station['empty_slots']}")
            print(f"     Timestamp: {station['timestamp']}")
            print()
    else:
        print("No OV-fiets stations found with 'Amsterdam' in their name.")
        print("Note: The 'OV-fiets' network covers all of the Netherlands, not just Amsterdam.")
else:
    print("❌ 'network_info' or 'stations' not available. Please run the previous cell first.")

🔍 Filtering OV-fiets stations for Amsterdam...
✅ Found 12 OV-fiets station(s) with 'Amsterdam' in their name:
   - Station: Amsterdam Noord (ID: 00deaf43155f1d6c57891732c5434185)
     Location: (52.4024, 4.9310)
     Bikes available: 7
     Docks available: None
     Timestamp: 2025-12-15T14:51:22.740364+00:00Z

   - Station: Amsterdam Sloterdijk (ID: 112f586b58911992ed883d933fdc06e6)
     Location: (52.3890, 4.8371)
     Bikes available: 32
     Docks available: None
     Timestamp: 2025-12-15T14:51:22.740385+00:00Z

   - Station: Amsterdam Centraal IJzijde West (ID: 3b405f6637f30d5c9fc2b87187c4a9e8)
     Location: (52.3801, 4.8988)
     Bikes available: 119
     Docks available: None
     Timestamp: 2025-12-15T14:51:22.740352+00:00Z

   - Station: Amsterdam Centraal Stationsplein  (ID: 49a79c775cd16466301f6b5055cfe4a9)
     Location: (52.3785, 4.8977)
     Bikes available: 452
     Docks available: None
     Timestamp: 2025-12-15T14:51:22.740347+00:00Z

   - Station: Amsterdam Zuid M

### 📊 Converting to DataFrame

Let's convert this JSON data into a pandas DataFrame for easier analysis:

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONVERT BIKE DATA TO DATAFRAME
# ═══════════════════════════════════════════════════════════

try:
    # Extract stations data
    stations = amsterdam_ov_stations

    # Create DataFrame
    bike_df = pd.DataFrame(
        [
            {
                'station_id': s['id'],
                'station_name': s['name'],
                'latitude': s['latitude'],
                'longitude': s['longitude'],
                'bikes_available': s['free_bikes'],
                'docks_available': s['empty_slots'],
                'timestamp': pd.to_datetime(s['timestamp'].replace('Z', '')) # Remove 'Z' for parsing
            }
            for s in stations
        ]
    )

    print("📊 Amsterdam OV-fiets Stations DataFrame:\n")
    print(bike_df.head())

    print(f"\n📈 Quick Statistics:")
    print(f"   Total Amsterdam OV-fiets stations: {len(bike_df)}")
    print(f"   Total bikes available: {bike_df['bikes_available'].sum()}")
    print(f"   Average bikes per station: {bike_df['bikes_available'].mean():.1f}")
    print(f"   Stations with no bikes: {(bike_df['bikes_available'] == 0).sum()}")

except Exception as e:
    print(f"❌ Error creating DataFrame: {e}")
    print(f"Note: Could not create DataFrame. This is okay for the demo!")
    print(f"We'll work with pre-downloaded data in later notebooks.")

📊 Amsterdam OV-fiets Stations DataFrame:

                         station_id                       station_name  \
0  00deaf43155f1d6c57891732c5434185                    Amsterdam Noord   
1  112f586b58911992ed883d933fdc06e6               Amsterdam Sloterdijk   
2  3b405f6637f30d5c9fc2b87187c4a9e8    Amsterdam Centraal IJzijde West   
3  49a79c775cd16466301f6b5055cfe4a9  Amsterdam Centraal Stationsplein    
4  5a638585dab37f4c290b87126127a643         Amsterdam Zuid Mahlerplein   

    latitude  longitude  bikes_available docks_available  \
0  52.402374   4.931019                7            None   
1  52.389030   4.837070               32            None   
2  52.380095   4.898784              119            None   
3  52.378482   4.897678              452            None   
4  52.336448   4.873438               36            None   

                         timestamp  
0 2025-12-15 14:51:22.740364+00:00  
1 2025-12-15 14:51:22.740385+00:00  
2 2025-12-15 14:51:22.740352+00:00  
3 20